In [4]:
# add path 
from pathlib import Path 
import sys 

# Dynamically resolve project root containing solver_config.py 
project_root = Path.cwd().resolve() 
while project_root != project_root.parent and not (project_root / "solver_config.py").exists(): 
    project_root = project_root.parent 

if str(project_root) not in sys.path: 
    sys.path.insert(0, str(project_root)) 

# set up 
from Set_up_general import * # for all relevant other files - general setup + figures 

Define format for table values


In [5]:
def format_val(val, format_type=".3g"):
    """
    Formats value to dynamic clean representation for LaTeX.
    """
    if val == 0:
        return "0"
    if isinstance(val, (int, np.integer)):
        return str(val)
    
    # Check if scientific notation is needed
    exponent = int(np.floor(np.log10(abs(val))))
    if -2 <= exponent <= 2:
        s = f"{val:{format_type}}"
        return s.rstrip('0').rstrip('.')
    else:
        s = f"{val:.2e}"
        s = s.replace("e-0", "e-").replace("e+0", "e+")
        coeff, exp = s.split('e')
        coeff = coeff.rstrip('0').rstrip('.')
        if coeff == "1":
            return f"1e{exp}"
        return f"{coeff}e{exp}"


(Tab. 1) - Table for Nominal Model Values

In [ ]:
# Format the plant values
plant_Kf = format_val(para_plant[0])
plant_kG = format_val(para_plant[1])
plant_kd = format_val(para_plant[2])

# Format the nominal model values dynamically  
n_sets = initial_para.shape[1]  
nom_vals = []  
for j in range(n_sets):  
    p_nom = 10**initial_para[:, j]  
    nom_vals.append([format_val(p_nom[i]) for i in range(3)])  

# Format controller model rows dynamically  
nom_rows = "\n".join([f"  contr. model {j+1} & {nom_vals[j][0]}  & {nom_vals[j][1]}& {nom_vals[j][2]}\\\\ " for j in range(n_sets)])  

# Construct and print the LaTeX table code  
latex_code = rf"""\begin{{table}}[h]
    \centering
    \caption{{Parameter for the plant and controller models.}}
    \label{{Tab: model parameters}}
    \begin{{tabular}}{{c|c|c|c}} 
  model & $K_{{ \text{{f}} }}$ (s\textsuperscript{{-1}}) & $k_{{G, \text{{eff}} }}$ (mm\,s\textsuperscript{{-1}}) &$k_{{ \text{{d}} }}$ (s\textsuperscript{{-1}})\\\hline
  plant model  & {plant_Kf}  & {plant_kG}& {plant_kd}\\\hline
{nom_rows}
    \end{{tabular}}
\end{{table}}"""  

print(latex_code)


\begin{table}[h]
    \centering
    \caption{Parameter for the plant and controller models.}
    \label{Tab: model parameters}
    \begin{tabular}{c|c|c|c} 
  model & $K_{ \text{f} }$ (s\textsuperscript{-1}) & $k_{G, \text{eff} }$ (mm\,s\textsuperscript{-1}) &$k_{ \text{d} }$ (s\textsuperscript{-1})\\\hline
  plant model  & 0.01  & 3.14e-3& 1e-3\\\hline
  contr. model 1 & 9e-3  & 1.57e-3& 1.1e-3\\ 
  contr. model 2 & 0.011  & 4.71e-3& 9e-4\\ 
  contr. model 3 & 9e-3  & 4.71e-3& 9e-4\\ 
  contr. model 4 & 0.011  & 1.57e-3& 1.1e-3\\ 
    \end{tabular}
\end{table}


(Tab. 2) - Table for Initial model conditions

In [ ]:
# Extract the values dynamically from the simulation arrays
v_nE0 = format_val(initial_cond[p.idx_E])
v_V0 = format_val(initial_cond[p.idx_V])
v_np0 = format_val(molar0_seeds)
v_N0 = format_val(N0[0])

# Construct and print the LaTeX table  
latex_code = rf"""\begin{{table}}[h]
    \centering
    \caption{{Initial conditions for the process simulation and data generation. If not stated otherwise the initial condition is zero.}}
    \label{{Tab: Init}}
    \begin{{tabular}}{{ c| c|| c| c}}
    initial state   & value & initial  state   & value \\\hline
    $n'^{{\bullet}}_{{ \text{{E}},0}}$ & {v_nE0} mmol &
    $V_{{0}}$ & {v_V0} L \\
    $ n''_ \text{{p,0}}$ & {v_np0} mmol &$ \mathcal{{N}}_ \text{{0}}$ & {v_N0}  
    \end{{tabular}}
\end{{table}}"""  

print(latex_code)


\begin{table}[h]
    \centering
    \caption{Initial conditions for the process simulation and data generation. If not stated otherwise the initial condition is zero.}
    \label{Tab: Init}
    \begin{tabular}{ c| c|| c| c}
    initial state   & value & initial  state   & value \\\hline
    $n'^{\bullet}_{ \text{E},0}$ & 0.453 mmol &
    $V_{0}$ & 0.1 L \\
    $ n''_ \text{p,0}$ & 1e-3 mmol &$ \mathcal{N}_ \text{0}$ & 3.1e+7  
    \end{tabular}
\end{table}


(Tab.4) - Table for Bounds for optimization

In [8]:
def format_latex_val(val, unit="", is_log_scale=False):
    """
    Formats bounds and variables into standard scientific/latex format.
    """
    if is_log_scale:
        exp = int(np.round(float(val)))
        val_str = f"1e{exp}"
    elif isinstance(val, (int, np.integer)):
        val_str = f"{val}"
    elif isinstance(val, (float, np.floating)):
        if abs(val) < 1e-3 or abs(val) >= 1e4:
            s = f"{val:.0e}"
            base, exp = s.split('e')
            exp = int(exp)
            if float(base) == 1.0:
                val_str = f"1e{exp}"
            else:
                val_str = f"{base}e{exp}"
        elif abs(val) >= 1:
            val_str = f"{val:.4f}".rstrip('0').rstrip('.')
        else:
            val_str = f"{val:.4f}".rstrip('0').rstrip('.')
    else:
        val_str = str(val)
    
    if unit:
        return f"{val_str} {unit}"
    return val_str

# Extract parameters dynamically from setup variables
lb_theta = format_latex_val(lb_para[0], is_log_scale=True)
ub_theta = format_latex_val(ub_para[0], is_log_scale=True)

lb_x = format_latex_val(X_lb_states[0], "mmol")
ub_x = format_latex_val(x_max[0], "mmol")

ub_V = format_latex_val(p.V_max, "L")
ub_L = format_latex_val(x_max[p.idx_L], "mm")
lb_u = "0"
ub_u = "1"

# Construct and print the LaTeX table
latex_code = f"""\\begin{{table}}[htbp]
    \\centering
    \\caption{{Parameters for the \\ac{{MHE}} and \\ac{{MPC}} algorithm. lb refers to lower bound, ub refers to upper bound. The bounds are given for the parameters $\\bm{{\\theta}}$, states $\\bm{{x}}$ and controls $\\bm{{u}}$. The bounds apply for all states, parameters and control if not stated otherwise.}}    
    \\begin{{tabular}}{{c|c||c|c}}
    parameter   & value &parameter   & value \\\\\\hline
      $\\textbf{{lb}}_{{\\hat{{\\bm{{\\theta}}}}}}$   & {lb_theta} &$\\textbf{{ub}}_{{\\hat{{\\bm{{\\theta}}}}}}$   & {ub_theta}\\\\
      $\\textbf{{lb}}_{{\\bm{{x}}}}$   & {lb_x} &$\\textbf{{ub}}_{{\\bm{{x}}}}$    &  {ub_x}\\\\
      $\\text{{ub}}_{{L}}$  & {ub_L}& $\\text{{ub}}_{{V}}$   & {ub_V} \\\\
      $\\textbf{{lb}}_{{\\bm{{u}}}}$  & {lb_u} &$\\textbf{{ub}}_{{\\bm{{u}}}}$  & {ub_u}
    \\end{{tabular}}
    \\label{{Tab:Opt para}}
\\end{{table}}"""

print(latex_code)


\begin{table}[htbp]
    \centering
    \caption{Parameters for the \ac{MHE} and \ac{MPC} algorithm. lb refers to lower bound, ub refers to upper bound. The bounds are given for the parameters $\bm{\theta}$, states $\bm{x}$ and controls $\bm{u}$. The bounds apply for all states, parameters and control if not stated otherwise.}    
    \begin{tabular}{c|c||c|c}
    parameter   & value &parameter   & value \\\hline
      $\textbf{lb}_{\hat{\bm{\theta}}}$   & 1e-5 &$\textbf{ub}_{\hat{\bm{\theta}}}$   & 1e0\\
      $\textbf{lb}_{\bm{x}}$   & -1 mmol &$\textbf{ub}_{\bm{x}}$    &  1e5 mmol\\
      $\text{ub}_{L}$  & 1000 mm& $\text{ub}_{V}$   & 0.5 L \\
      $\textbf{lb}_{\bm{u}}$  & 0 &$\textbf{ub}_{\bm{u}}$  & 1
    \end{tabular}
    \label{Tab:Opt para}
\end{table}


Appendix - Tab.  7 All parameter values

In [ ]:
# 1. Enzyme Activity
# Read from process parameters
U_E0 = p.U_ml  # U ml^-1
A_E0 = p.A_E   # U mg^-1

# 2. Saturation Concentrations
c_sat_d = format_val(p.Cs_IPA_3DPPA)
c_sat_p = format_val(p.Cs_3MPEA_3DPPA)
c_sat_3MAP = format_val(p.Cs_3MAP)

# 3. Molar Weights
M_w_E = f"{p.M_E:.4f}"
M_w_p = f"{p.M_3MPEA_3DPPA * 1e3:.4f}e-3"
M_w_ACE = f"{p.M_Ac * 1e3:.2f}e-3"
M_w_IPA = f"{p.M_IPA * 1e3:.2f}e-3"
M_w_3DPPA = f"{p.M_3DPPA * 1e3:.2f}e-3"
M_w_3MAP = f"{p.M_3MAP * 1e3:.2f}e-3"

# 4. Densities
# Conversion note: In physical chemistry, density is given in g L^-1 or kg m^-3 (which equals g L^-1).
# We print values in g L^-1 (which is the actual unit of the values 1217.8, 790, etc.)
rho_p = f"{p.density * 1e6:.1f}"
rho_ACE = f"{p.rho_l_Ac:.0f}"
rho_IPA = f"{p.rho_IPA:.0f}"
rho_3DPPA = f"{p.rho_3DPPA:.0f}"
rho_3MAP = f"{p.rho_3MAP:.0f}"

# 5. Aspect Ratios
AR_l_w = format_val(p.shape_P1)
AR_w_d = format_val(p.shape_P2)

# 6. Kinetic Parameters
K_m_Ac = format_val(p.K_m_Ac)
K_m_IPA = format_val(p.K_m_IPA)
K_m_3MAP = format_val(p.K_m_MAP)
K_i_IPA = format_val(p.K_i_IPA)
K_m_3MPEA = format_val(p.K_m_MPEA)
K_i_3MPEA = format_val(p.K_i_MPEA)
K_eq = format_val(p.K_eq)

# 7. Reaction Rates
k_3MAP = format_val(p.k_3MAP)
k_d = format_val(p.k_d)

# 8. Physical/Simulation Constants
V_t0 = format_val(initial_cond[p.idx_V])
N_A = f"{p.N_A:.8e}"
Z_star = format_val(p.Z_p)
V_UC = "1029.43(1e-7)\\textsuperscript{3}"
N0_val = f"$\\approx$ {format_val(N0[0])}"
n_p_0 = format_val(molar0_seeds)
L0_3 = format_val(p.initial_L3)

# Construct and print the LaTeX table  
latex_code = rf"""\begin{{table}}[htbp]
    \centering
    \caption{{Parameters for the process simulation and data generation. Parameters marked by $\star$ are taken from Al-Haque et al. \cite{{Al-Haque12}}. Parameters marked with a star $\star$ are taken from \cite{{Neuburger2021}}. The molar weight of the enzyme $ M_\text{{w,E}}$ is taken from \cite{{Toro2009}}. }}
    \label{{Tab Appendix: Parameters}}
    \begin{{tabular}}{{ c| c }}   
    parameter   & value \\\hline
    % Enzyme activity
    $U_\text{{E,0}}$ & {U_E0} U\,ml\textsuperscript{{-1}}\\
    $A_\text{{E,0}}$ & {A_E0} U\,mg\textsuperscript{{-1}}\\
    \hline
    % Saturation concentrations
    $c_\text{{sat,d}}$ & {c_sat_d} mmol\,L\textsuperscript{{-1}}  \\
    $c_\text{{sat,p}}$ & {c_sat_p} mmol\,L\textsuperscript{{-1}} \\
    $c_\text{{sat,3MAP}}$ & {c_sat_3MAP} mmol\,L\textsuperscript{{-1}} \\
    \hline
    % Molar weights
    $M_\text{{w,E}}$   & {M_w_E} g\,mmol\textsuperscript{{-1}}  \\
    $M_\text{{w,p}}$   & {M_w_p} g\,mmol\textsuperscript{{-1}}  \\
    $M_\text{{w,ACE}}$   &  {M_w_ACE} g\,mmol\textsuperscript{{-1}}  \\
    $M_\text{{w,IPA}}$   &  {M_w_IPA} g\,mmol\textsuperscript{{-1}}  \\        
    $M_\text{{w,3DPPA}}$   &  {M_w_3DPPA} g\,mmol\textsuperscript{{-1}}  \\  
    $M_\text{{w,3MAP}}$   &  {M_w_3MAP} g\,mmol\textsuperscript{{-1}}  \\ 
    \hline
    % Densities (in g L^-1 or kg m^-3)
    $\rho_\text{{p}}$ & {rho_p} g\,L\textsuperscript{{-1}} \\
    $\rho_\text{{ACE}}$ & {rho_ACE} g\,L\textsuperscript{{-1}} \\
    $\rho_\text{{IPA}}$ &  {rho_IPA} g\,L\textsuperscript{{-1}} \\        
    $\rho_\text{{3DPPA}}$ &  {rho_3DPPA} g\,L\textsuperscript{{-1}} \\  
    $\rho_\text{{3MAP}}$ &  {rho_3MAP} g\,L\textsuperscript{{-1}} \\ 
    \hline
    % Aspect ratios
    $\text{{AR}}_\text{{l,w}}$ & {AR_l_w} \\
    $\text{{AR}}_\text{{w,d}}$ & {AR_w_d} \\
    \hline
    % Kinetic parameters
    $K_{{\text{{m,Ac}}}}^\star$ & {K_m_Ac} mM \\
    $K_{{\text{{m,IPA}}}}^\star$ & {K_m_IPA} mM \\
    $K_{{\text{{m,3MAP}}}}^\star$& {K_m_3MAP} mM \\
    $K_{{\text{{i,IPA}}}}^\star$ & {K_i_IPA} mM \\
    $K_{{\text{{m,3MPEA}}}}^\star$ & {K_m_3MPEA} mM \\
    $K_{{\text{{i,3MPEA}}}}^\star$ & {K_i_3MPEA} mM \\
    $K_{{\text{{eq}}}}^\star$& {K_eq}  \\
    \hline
    % Reaction rates
    $k_\text{{3MAP}}$& {k_3MAP} s\textsuperscript{{-1}} \\
    $k_\text{{d}}$& {k_d} s\textsuperscript{{-1}}\\
    \hline
    % Physical/Simulation constants
    $V(t_0)$& {V_t0} L \\
    $N_A$ & {N_A} mmol\textsuperscript{{-1}} \\
    $Z^\star$ & {Z_star}\\
    $V_{{UC}}^\star$ & {V_UC} mm\textsuperscript{{3}} \\
    $\mathcal{{N}}_0$ & {N0_val} \\
    $n_{{\text{{p}},0}}^{{\prime\prime}}$ & {n_p_0} mmol \\
    $L_0^3$& {L0_3} mm\textsuperscript{{3}}
    \end{{tabular}}
\end{{table}}"""  

print(latex_code)


\begin{table}[htbp]
    \centering
    \caption{Parameters for the process simulation and data generation. Parameters marked by $\star$ are taken from Al-Haque et al. \cite{Al-Haque12}. Parameters marked with a star $\star$ are taken from \cite{Neuburger2021}. The molar weight of the enzyme $ M_\text{w,E}$ is taken from \cite{Toro2009}. }
    \label{Tab Appendix: Parameters}
    \begin{tabular}{ c| c }   
    parameter   & value \\\hline
    % Enzyme activity
    $U_\text{E,0}$ & 80 U\,ml\textsuperscript{-1}\\
    $A_\text{E,0}$ & 170 U\,mg\textsuperscript{-1}\\
    \hline
    % Saturation concentrations
    $c_\text{sat,d}$ & 55 mmol\,L\textsuperscript{-1}  \\
    $c_\text{sat,p}$ & 4.7 mmol\,L\textsuperscript{-1} \\
    $c_\text{sat,3MAP}$ & 15 mmol\,L\textsuperscript{-1} \\
    \hline
    % Molar weights
    $M_\text{w,E}$   & 0.1039 g\,mmol\textsuperscript{-1}  \\
    $M_\text{w,p}$   & 377.4790e-3 g\,mmol\textsuperscript{-1}  \\
    $M_\text{w,ACE}$   &  58.08e-3 g\,mmol\textsuper

Appendix - Tab.  8 STD values

In [ ]:
# Extract and format standard deviations dynamically
std_ipa = format_latex_val(std_mess[0], "mmol")
std_3mpea = format_latex_val(std_mess[1], "mmol")
std_ace = format_latex_val(std_mess[2], "mmol")
std_3map_p = format_latex_val(std_mess[3], "mmol")
std_3dppa = format_latex_val(std_mess[4], "mmol")
std_e = format_latex_val(std_mess[5], "mmol")
std_3map_np = format_latex_val(std_mess[6], "mmol")
std_d = format_latex_val(std_mess[7], "mmol")
std_V = format_latex_val(std_mess[8], "L")
std_p = format_latex_val(std_mess[9], "mmol")

# Construct and print the LaTeX table  
latex_code = rf"""\begin{{table}}[htbp]
    \centering
    \caption{{\acp{{STD}} for the artificial data generation}}
    \label{{Tab: Appendix std}}
    \begin{{tabular}}{{ c| c|| c| c}}
     state & std & state & std \\\hline
     $n'^{{\bullet}}_{{ \text{{IPA}},0}}$ &  {std_ipa} & $n'^{{\bullet}}_{{ \text{{3MPEA}},0}}$ &  {std_3mpea}\\
    $n'^{{\bullet}}_{{ \text{{ACE}},0}}$ &  {std_ace} & $n'^{{\bullet}}_{{ \text{{3MAP}},0}}$ &  {std_3map_p}\\
    $n'^{{\bullet}}_{{ \text{{3DPPA}},0}}$ &  {std_3dppa} & $n'^{{\bullet}}_{{ \text{{E}},0}}$ &  {std_e} \\
    $n'^{{\circ}}_{{ \text{{3MAP}},0}}$   &  {std_3map_np} & $n''_{{ \text{{d}},0}}$ &  {std_d} \\
    $V_{{0}}$ &  {std_V} & $n''_{{ \text{{p}},0}}$ &  {std_p}
    \end{{tabular}}
\end{{table}}"""  

print(latex_code)


\begin{table}[htbp]
    \centering
    \caption{\acp{STD} for the artificial data generation}
    \label{Tab: Appendix std}
    \begin{tabular}{ c| c|| c| c}
     state & std & state & std \\\hline
     $n'^{\bullet}_{ \text{IPA},0}$ &  0.0691 mmol & $n'^{\bullet}_{ \text{3MPEA},0}$ &  0.0012 mmol\\
    $n'^{\bullet}_{ \text{ACE},0}$ &  1e-5 mmol & $n'^{\bullet}_{ \text{3MAP},0}$ &  0.0187 mmol\\
    $n'^{\bullet}_{ \text{3DPPA},0}$ &  0.0702 mmol & $n'^{\bullet}_{ \text{E},0}$ &  0.0045 mmol \\
    $n'^{\circ}_{ \text{3MAP},0}$   &  0.0446 mmol & $n''_{ \text{d},0}$ &  0.043 mmol \\
    $V_{0}$ &  0.0013 L & $n''_{ \text{p},0}$ &  0.0016 mmol
    \end{tabular}
\end{table}
